In [ ]:
from dotenv import load_dotenv  # loads variables from .env file into os.environ
import os                        # gives access to environment variables

load_dotenv()                            # reads .env in current directory
api_key = os.getenv("ANTHROPIC_API_KEY") # retrieves the key value (None if missing)
print("API key loaded:", api_key is not None)  # True = key found, False = check .env

In [ ]:
from anthropic import Anthropic  # imports the official Anthropic Python SDK client class

client = Anthropic(api_key=api_key)  # instantiates client; api_key authenticates every request
print("Anthropic client ready")       # confirms object creation succeeded

In [ ]:
response = client.messages.create(       # calls the Messages API endpoint
    model="claude-haiku-4-5",            # fastest/cheapest model — good for smoke tests
    #    model="claude-sonnet-4-5",      # swap here for higher-quality responses
    max_tokens=50,                       # hard cap on output length; required by the API
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]  # minimal valid messages list
)
print(response.content[0].text)  # content is a list of blocks; [0] is the first text block

# CCA Lab: System Prompts Exercise

**Course:** Building with the Claude API  
**Sub-module:** Accessing Claude with the API  
**Lesson:** System Prompts Exercise

## What this lab covers
- Passing a system prompt as a keyword argument to a `chat()` helper function
- Defining a role persona (Python engineer) via the `system` parameter
- Observing how the system prompt changes Claude's response style and format
- Comparing responses with vs. without a system prompt on the same coding task
- The `system=None` anti-pattern and why correct omission matters
- Improvement cycle: write → evaluate → improve → re-evaluate with a rubric
- Failure mode analysis and token usage tracking

## CCA Domains Exercised
- **Primary:** Prompt Engineering
- **Contributing:** Agentic Architecture (error patterns)

## Learning Objectives
1. Explain what the `system` parameter does and when to use it.
2. Pass a role persona via `system` to constrain tone, style, and output format.
3. Demonstrate the behavioral difference between prompted and un-prompted responses.
4. Identify and avoid the `system=None` anti-pattern.
5. Score responses with a deterministic rubric and close the improvement loop.

## Section 1: Prerequisites, Environment & API Glossary
### CCA objective demonstrated: Identify every Messages API parameter and its role

| Parameter | Type | Required | Purpose |
|-----------|------|----------|---------|
| `model` | string | ✅ | Which Claude model to invoke |
| `max_tokens` | int | ✅ | Hard cap on output tokens — API rejects if omitted |
| `messages` | list[dict] | ✅ | Alternating `user`/`assistant` turns |
| `system` | string | ❌ | Persistent developer instructions sent before the conversation |
| `temperature` | float 0-1 | ❌ | Randomness of sampling (default 1.0) |
| `stop_sequences` | list[str] | ❌ | Tokens that halt generation early |

**Installed packages needed:** `anthropic`, `python-dotenv`  
**`.env` file needed:** `ANTHROPIC_API_KEY=sk-ant-...`

## Section 2: ask() Helper — Full Statement-by-Statement Annotation
### CCA objective demonstrated: Build a reusable chat() wrapper that handles system prompt omission correctly

In [ ]:
# --- Token usage log — appended after every API call for Section 9 analysis ---
usage_log = []  # list of dicts; each entry records one API call's token counts

def ask(prompt: str, system: str = None, model: str = "claude-haiku-4-5", max_tokens: int = 512) -> str:
    """
    Single-turn convenience wrapper around client.messages.create().

    Parameters
    ----------
    prompt     : The user message text to send.
    system     : Optional system prompt string (omitted entirely when None).
    model      : Claude model ID string.
    max_tokens : Hard upper bound on response length in tokens.

    Returns
    -------
    str  — The assistant's reply text.
    """
    # Build the keyword arguments dict dynamically so we never pass system=None.
    # The API interprets system=None as an empty string, which is a different
    # behavioral signal than omitting the parameter entirely.
    kwargs = {}                          # start with no extra kwargs
    if system is not None:               # only add 'system' key when a real string is provided
        kwargs["system"] = system        # inject system prompt into kwargs dict

    response = client.messages.create(  # make the synchronous HTTP request to the Messages API
        model=model,                    # which Claude model to use (haiku=fast, sonnet=quality)
        max_tokens=max_tokens,          # required: API raises InvalidRequestError if omitted
        messages=[{"role": "user",      # messages must be a list of role/content dicts
                   "content": prompt}], # content holds the actual text of this turn
        **kwargs                        # unpacks system= only when it was added above
    )
    # response.content is a LIST of content blocks (text, tool_use, etc.).
    # We access [0] because single-turn calls virtually always return one text block.
    # response.content[0].text extracts the plain string from that block object.
    text = response.content[0].text     # the assistant's reply as a plain Python string

    # response.stop_reason tells us WHY generation stopped:
    #   "end_turn"   — model finished naturally (most common)
    #   "max_tokens" — hit the token ceiling; response may be truncated
    #   "stop_sequence" — a stop_sequence token was encountered
    stop = response.stop_reason         # capture stop_reason for logging

    # Append a usage record so Section 9 can build a per-call table.
    usage_log.append({                           # dict holding this call's metadata
        "call": len(usage_log) + 1,              # sequential call number (1-based)
        "label": prompt[:40].replace("\n"," "),  # truncated prompt for readability
        "input_tokens":  response.usage.input_tokens,   # tokens sent to the model
        "output_tokens": response.usage.output_tokens,  # tokens generated by the model
        "stop_reason":   stop                    # why generation ended
    })

    return text  # hand the reply string back to the caller


def chat(messages: list, system: str = None, model: str = "claude-haiku-4-5", max_tokens: int = 512) -> str:
    """
    Multi-turn wrapper: accepts a full messages list (for conversation history).

    Parameters
    ----------
    messages   : List of {role, content} dicts accumulated across turns.
    system     : Optional system prompt (omitted when None — never passed as None).
    model      : Claude model ID.
    max_tokens : Token ceiling for this response.

    Returns
    -------
    str  — Assistant reply text; also appends an assistant turn to `messages`.
    """
    kwargs = {}                    # build extra kwargs conditionally
    if system is not None:         # guard: only add system when a real string given
        kwargs["system"] = system  # production pattern: system in kwargs, not messages list

    response = client.messages.create(  # API call with the full messages history
        model=model,
        max_tokens=max_tokens,
        messages=messages,         # send entire conversation history each turn
        **kwargs                   # conditionally injects system=
    )

    reply = response.content[0].text  # extract text from first content block

    # Append assistant reply to messages so future turns have full context.
    messages.append({"role": "assistant", "content": reply})

    usage_log.append({                           # log token usage for this call
        "call": len(usage_log) + 1,
        "label": (messages[0]["content"] if messages else "")[:40].replace("\n"," "),
        "input_tokens":  response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "stop_reason":   response.stop_reason
    })

    return reply  # return reply to caller; messages list already updated in place


def add_user_message(messages: list, content: str) -> None:
    """
    Appends a user turn to the messages list in place.

    Parameters
    ----------
    messages : The running conversation messages list.
    content  : The user's message text.
    """
    messages.append({"role": "user", "content": content})  # extend list with new user dict


print("ask(), chat(), and add_user_message() helpers defined.")  # confirm all helpers ready

## Section 3: D-SYS — System Prompt: Role, Behavior Contract, and Production Pattern
### CCA objective demonstrated: Use the system parameter to define a role persona and behavioral contract

The `system` parameter is the **developer's persistent voice** — it sets the behavioral
contract for every turn in the conversation without being visible to the end user.

### Decision Table: system parameter vs. user turn

| What belongs in `system` | What belongs in the `user` turn |
|--------------------------|----------------------------------|
| Role / persona definition (`"You are a Python engineer…"`) | The specific task or question for this request |
| Tone and style constraints (`"Be concise, avoid prose explanations"`) | Dynamic runtime data (filenames, user input) |
| Output format rules (`"Always return a function with a docstring"`) | Follow-up questions or clarifications |
| Invariant safety / policy guardrails | Turn-specific context that changes per call |
| Persona persistence across all turns | One-off instructions not needed again |

**Architectural principle:** Everything that must remain constant across every conversation turn belongs in `system`; everything that changes turn-by-turn belongs in the `messages` list.

---
We will now run **the same coding task** with and without a system prompt so you can
see exactly how the system parameter changes Claude's response style.

In [ ]:
# --- The canonical system prompt exercise pattern ---

TASK = (                                        # the user message we'll send both times
    "Write a Python function that checks a "
    "string for duplicate characters."
)

SYSTEM_PROMPT = (                               # the role persona we inject via system=
    "You are a Python engineer who writes very concise code. "
    "Return only the function — no prose explanation."
)

# --- Without system prompt ---
print("=" * 60)
print("WITHOUT system prompt (default Claude behavior):")
print("=" * 60)
response_no_sys = ask(TASK)                     # ask() omits system entirely when not provided
print(response_no_sys)                          # typically verbose: explanation + code + walkthrough

print()

# --- With system prompt (the canonical exercise pattern) ---
print("=" * 60)
print("WITH system prompt (Python engineer persona):")
print("=" * 60)

messages = []                                   # Step 1: initialize empty messages list
add_user_message(messages, TASK)                # Step 2: append user turn to messages
response_with_sys = chat(                       # Step 3: call chat() with system= persona
    messages,
    system=SYSTEM_PROMPT                        # system= constrains tone, style, and format
)
print(response_with_sys)                        # expect: concise function, docstring, no prose

print()
print("--- Observation ---")
print("No-system response length (chars):", len(response_no_sys))   # likely much longer
print("System-prompted length (chars):   ", len(response_with_sys)) # concise by design

## Section 4: D-ERR — Intentional Error Demonstration: system=None Anti-Pattern
### CCA objective demonstrated: Recognize and avoid the system=None anti-pattern in production code

Passing `system=None` explicitly is a **silent bug** — the API may accept it but the
behavior is undefined or implementation-dependent. The correct production pattern is
to **omit the parameter entirely** when no system prompt is needed.

In [ ]:
import anthropic  # needed for exception type references below

SIMPLE_PROMPT = "What is 2 + 2?"  # minimal prompt for error demonstration

# --- Anti-pattern: explicitly passing system=None ---
print("=" * 60)
print("ANTI-PATTERN: system=None passed explicitly")
print("=" * 60)
try:
    # This bypasses our safe kwargs guard because we're calling the API directly.
    # system=None is either rejected or silently ignored — never do this in production.
    bad_response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=50,
        messages=[{"role": "user", "content": SIMPLE_PROMPT}],
        system=None                # ← anti-pattern: None is not a valid system string
    )
    # If the SDK silently coerces None to "", we print a warning instead of crashing.
    print("SDK silently coerced system=None. Reply:", bad_response.content[0].text)
    print("⚠️  WARNING: system=None was accepted but is NOT the correct omission pattern.")
    print("   This can produce unpredictable behavior across SDK versions.")
except anthropic.BadRequestError as e:
    # Some SDK versions raise BadRequestError when system is explicitly None.
    print(f"❌ BadRequestError caught (expected): {e}")
    print("   Root cause: system=None is not a valid string value.")
except Exception as e:
    # Catch any other unexpected error type and report it clearly.
    print(f"❌ Unexpected error type {type(e).__name__}: {e}")

print()

# --- Correct pattern: omit system entirely using our safe ask() wrapper ---
print("=" * 60)
print("CORRECT PATTERN: system omitted entirely via ask() wrapper")
print("=" * 60)
good_response = ask(SIMPLE_PROMPT)   # ask() only injects system= when a real string is given
print("Reply:", good_response)        # clean response, no system parameter sent at all
print("✅ system parameter omitted entirely — this is the production pattern.")

print()
print("--- Why this matters ---")
print("The ask()/chat() helpers use `if system is not None` to guard the kwargs dict.")
print("This guarantees system= is either a meaningful string or absent from the request.")

## Section 5: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
### CCA objective demonstrated: Apply a deterministic rubric to iteratively improve system prompts

We score system-prompted responses on four dimensions and use the rubric to guide
incremental improvements.

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs.

In [ ]:
import re  # regex module for deterministic keyword scoring

def score_response(text: str) -> int:
    """
    Scores a Python code response on four rubric dimensions (0-4 total).

    Rubric dimensions:
      D1 — Contains a function definition        (1 pt)
      D2 — Has a docstring                       (1 pt)
      D3 — Uses a set for O(n) deduplication     (1 pt)
      D4 — No verbose prose paragraph            (1 pt, penalised if prose > 100 chars outside code)

    Parameters
    ----------
    text : The assistant's reply string.

    Returns
    -------
    int  — Score in [0, 4].
    """
    # D1: Check for a function definition keyword.
    # re.search scans the whole string (not just the start like re.match).
    # \b word-boundary anchors prevent matching 'def' inside words like 'define'.
    d1 = bool(re.search(r"\bdef\b", text))       # True if 'def' keyword found anywhere
    d1_score = int(d1)                            # int(True)=1, int(False)=0
    icon1 = "✅" if d1 else "❌"                   # pass/fail icon for readability
    print(f"{icon1} D1 — function def present:   +{d1_score}")  # print per-dimension result

    # D2: Check for a docstring (triple-quoted string inside the function).
    d2 = bool(re.search(r'\"\"\"', text))         # looks for triple-double-quote markers
    d2_score = int(d2)
    icon2 = "✅" if d2 else "❌"
    print(f"{icon2} D2 — docstring present:      +{d2_score}")

    # D3: Check for set usage — the idiomatic O(n) deduplication structure.
    # \bset\b matches the keyword; also allow set() call form.
    d3 = bool(re.search(r"\bset\b", text))        # True if set literal or set() call used
    d3_score = int(d3)
    icon3 = "✅" if d3 else "❌"
    print(f"{icon3} D3 — set for deduplication:  +{d3_score}")

    # D4: Penalise verbose prose. Strip code blocks first, then measure remaining text.
    # re.sub replaces code-fence blocks with empty string, leaving only prose.
    prose_only = re.sub(r"```[\s\S]*?```", "", text)  # remove ```...``` fenced blocks
    d4 = len(prose_only.strip()) < 120            # pass if remaining prose is short (<120 chars)
    d4_score = int(d4)
    icon4 = "✅" if d4 else "❌"
    print(f"{icon4} D4 — concise (no verbose prose): +{d4_score}")

    total = d1_score + d2_score + d3_score + d4_score  # sum all dimensions
    print(f"   ─────────────────────────────")
    print(f"   TOTAL SCORE: {total} / 4")
    return total  # return numeric score for comparison table


print("score_response() rubric defined — 4 dimensions, 1 pt each.")

In [ ]:
# ── Improvement Cycle ─────────────────────────────────────────────────────────
# Pattern: write system prompt → evaluate → improve → re-evaluate

CODING_TASK = (                                  # fixed user message for all versions
    "Write a Python function that checks a string for duplicate characters."
)

# ── Version 1: No system prompt (baseline) ────────────────────────────────────
print("VERSION 1 — No system prompt (baseline)")
print("-" * 50)
v1_reply = ask(CODING_TASK)                      # call without any system prompt
v1_score = score_response(v1_reply)              # evaluate with rubric
print()

# ── Version 2: Basic role persona ─────────────────────────────────────────────
print("VERSION 2 — Basic role persona")
print("-" * 50)
sys_v2 = "You are a Python engineer who writes very concise code."  # minimal persona
v2_reply = ask(CODING_TASK, system=sys_v2)       # inject role persona via system=
v2_score = score_response(v2_reply)              # evaluate same rubric
print()

# ── Version 3: Improved system prompt with explicit format constraints ─────────
print("VERSION 3 — Explicit format + structure constraints")
print("-" * 50)
sys_v3 = (                                       # more precise behavioral contract
    "You are a Python engineer who writes very concise code. "
    "Always include a docstring. Use a set for O(n) deduplication. "
    "Return only the function — no prose explanation, no markdown fences."
)
v3_reply = ask(CODING_TASK, system=sys_v3)       # improved system prompt applied
v3_score = score_response(v3_reply)              # evaluate to confirm improvement
print()

# ── Side-by-side comparison table ─────────────────────────────────────────────
PASS_THRESHOLD = 4                               # score >= 4 is a PASS (all criteria required)
print("=" * 70)
print("IMPROVEMENT CYCLE — COMPARISON TABLE")
print("=" * 70)

# Build rows before f-string to avoid backslash-in-expression SyntaxError (Python 3.11+)
rows = [
    ("v1 (no system)",   sys_v3[:0] or "(none)"[:30],     v1_score),
    ("v2 (role only)",   sys_v2[:30],                      v2_score),
    ("v3 (full contract)",sys_v3[:30],                     v3_score),
]

print(f"{'Version':<22} {'System prompt (truncated)':<32} {'Score':>5} {'Status':>8}")
print("-" * 70)
for version, snippet, score in rows:             # iterate over pre-built list (no backslash needed)
    status = "PASS" if score >= PASS_THRESHOLD else "FAIL"  # threshold check per version
    print(f"{version:<22} {snippet:<32} {score:>5} {status:>8}")

print("=" * 70)
best_score = max(v1_score, v2_score, v3_score)   # find highest score across all versions
final = "✅ PASS" if best_score >= PASS_THRESHOLD else "❌ FAIL"  # overall result
print(f"Best score: {best_score}/4  →  {final} (threshold = {PASS_THRESHOLD}/4)")

## Section 6: Failure Mode Analysis
### CCA objective demonstrated: Identify, describe, and recover from common system prompt failure modes

| Failure Mode | Trigger | Symptom |
|---|---|---|
| **system=None anti-pattern** | Passing `system=None` explicitly to the API | SDK may raise `BadRequestError` or silently coerce to empty string; behavior is SDK-version dependent |
| **Overly vague persona** | System prompt says only `"Be helpful"` | Response ignores intended format/tone constraints; output is indistinguishable from no-system-prompt behavior |
| **Conflicting instructions** | System says "no prose" but user turn says "explain everything" | Claude follows the more recent user instruction; system-level constraints are overridden |
| **System prompt in messages list** | Developer puts `{"role": "system", "content": "…"}` in `messages` | API raises `InvalidRequestError` — `"system"` is not a valid role in the messages array |
| **Missing max_tokens** | `client.messages.create()` called without `max_tokens` | API raises `InvalidRequestError` immediately — `max_tokens` is a required parameter |

In [ ]:
# --- Live failure demo: system role injected into messages list ---
# This is the most common beginner mistake when coming from OpenAI's API,
# which does accept {"role": "system"} in the messages array.

print("FAILURE DEMO 1: system role in messages list")
print("-" * 50)
try:
    client.messages.create(                       # direct API call to expose the raw error
        model="claude-haiku-4-5",
        max_tokens=50,
        messages=[
            {"role": "system",                    # ← WRONG: 'system' is not a valid messages role
             "content": "You are a Python engineer."},
            {"role": "user", "content": "Write hello world."}
        ]
    )
except anthropic.BadRequestError as e:
    print(f"❌ BadRequestError caught (expected): {e}")
    print("   FIX: Pass system as a keyword argument, not as a messages entry.")
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

print()

# --- Live failure demo: missing max_tokens ---
print("FAILURE DEMO 2: missing max_tokens (required parameter)")
print("-" * 50)
try:
    client.messages.create(                       # call without max_tokens
        model="claude-haiku-4-5",
        messages=[{"role": "user", "content": "Hello"}]
        # max_tokens omitted — API will reject this
    )
except anthropic.BadRequestError as e:
    print(f"❌ BadRequestError caught (expected): {e}")
    print("   FIX: Always include max_tokens — it is a required parameter.")
except TypeError as e:
    # SDK may raise TypeError before the HTTP call if it validates locally.
    print(f"❌ TypeError caught (SDK-level validation): {e}")
    print("   FIX: Always include max_tokens in every client.messages.create() call.")
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

print()
print("✅ Both failure modes demonstrated and caught with descriptive messages.")

## Section 7: Token Usage Tracking
### CCA objective demonstrated: Monitor per-call and cumulative token consumption across the full session

In [ ]:
# --- Build per-call usage table from usage_log populated throughout the notebook ---
# Every ask() and chat() call appended a record; we now summarise the full session.

print("TOKEN USAGE — PER-CALL TABLE")
print("=" * 80)
# Header row with aligned columns
print(f"{'Call':>4}  {'Label (truncated)':<42}  {'Input':>7}  {'Output':>7}  {'Cumul In':>9}  {'Cumul Out':>9}")
print("-" * 80)

cumul_in  = 0   # running total of input tokens across all calls
cumul_out = 0   # running total of output tokens across all calls

for entry in usage_log:                          # iterate over every logged API call
    cumul_in  += entry["input_tokens"]           # accumulate input token count
    cumul_out += entry["output_tokens"]          # accumulate output token count
    label = entry["label"][:42]                  # truncate label to fit column (backslash-safe)
    print(
        f"{entry['call']:>4}  "
        f"{label:<42}  "
        f"{entry['input_tokens']:>7}  "
        f"{entry['output_tokens']:>7}  "
        f"{cumul_in:>9}  "
        f"{cumul_out:>9}"
    )

print("=" * 80)
print(f"{'TOTALS':<50}  {cumul_in:>7}  {cumul_out:>7}")
print()
print(f"Total API calls this session : {len(usage_log)}")
print(f"Total input  tokens          : {cumul_in}")
print(f"Total output tokens          : {cumul_out}")
print(f"Total tokens (in + out)      : {cumul_in + cumul_out}")
print()
# Insight: system prompts add to input token count every turn — factor this
# into cost estimates for high-throughput production applications.
print("⚠️  Note: longer system prompts increase input_tokens on every API call.")
print("   Use prompt caching (beta) for repeated long system prompts in production.")

## Section 8: Practice Drills
### CCA objective demonstrated: Apply system prompt patterns independently to new scenarios

In [ ]:
# ── Drill 1: Customer Support Agent ───────────────────────────────────────────
# Task: Write a system prompt that makes Claude behave as a polite,
# solution-focused customer support agent for a SaaS product.
# Then ask it to handle a refund request and observe the tone change.

print("DRILL 1 — Customer Support Agent persona")
print("-" * 50)

support_system = (                               # YOUR system prompt goes here
    "You are a polite, empathetic customer support agent for CloudDash SaaS. "
    "Always acknowledge the customer's concern, offer one clear solution, "
    "and end with an offer for further help. Be concise — 3 sentences max."
)

support_reply = ask(
    "I was charged twice this month and I want a refund immediately.",
    system=support_system                        # persona constrains tone and structure
)
print(support_reply)
print()

# ── Drill 2: Data Extraction Agent ────────────────────────────────────────────
# Task: Write a system prompt that constrains Claude to output ONLY valid JSON
# for structured data extraction — no prose, no markdown fences.

print("DRILL 2 — Structured JSON extraction persona")
print("-" * 50)

extractor_system = (                             # pure-JSON output contract
    "You are a data extraction engine. "
    "Output ONLY a valid JSON object with keys: name, amount, currency, date. "
    "No prose, no markdown, no explanation."
)

extractor_reply = ask(
    "Invoice: Alice Smith, $1,250.00 USD, issued 2024-03-15.",
    system=extractor_system                      # strict output format via system=
)
print(extractor_reply)                           # should be pure JSON
print()

# ── Drill 3: Persona Comparison ───────────────────────────────────────────────
# Task: Send the SAME question to two different personas and observe the difference.

print("DRILL 3 — Same question, two personas")
print("-" * 50)

question = "What is recursion?"                  # identical user message for both

sys_expert = "You are a CS professor. Give a rigorous one-paragraph technical definition."
sys_child  = "You are explaining to a 10-year-old. Use a fun analogy, 2 sentences max."

reply_expert = ask(question, system=sys_expert)  # expert persona
reply_child  = ask(question, system=sys_child)   # child-friendly persona

print("Expert answer:")
print(reply_expert)
print()
print("Child-friendly answer:")
print(reply_child)
print()
print("✅ All 3 drills complete. Observe how system= alone reshapes style, tone, and format.")

## Section 9: CCA Takeaways — 5 Exam-Ready Mental Models
### CCA objective demonstrated: Consolidate system prompt knowledge into exam-applicable principles

| # | Mental Model | Key Rule | Exam Trigger |
|---|---|---|---|
| 1 | **system = developer voice** | The `system` parameter is the persistent behavioral contract set by the developer, not the user. | Any question about where to place role/persona instructions. |
| 2 | **Omit, never None** | When no system prompt is needed, omit the parameter entirely. Never pass `system=None` — use `if system is not None` guard in wrappers. | Questions about the `system=None` anti-pattern or conditional parameter inclusion. |
| 3 | **system ≠ messages[role=system]** | The Anthropic API does not accept `{"role": "system"}` in the `messages` list. System instructions live in the `system` keyword argument only. | Questions distinguishing Anthropic vs. OpenAI API conventions. |
| 4 | **System constrains all turns** | The system prompt applies to every turn in the conversation — you write it once, it persists. Repeating instructions in every user message is an anti-pattern. | Questions about multi-turn conversation design and token efficiency. |
| 5 | **Cost × system prompt length** | System prompt tokens are counted as input tokens on **every** API call. Long system prompts in high-throughput apps should use prompt caching to reduce cost. | Questions about token cost optimisation and production deployment. |